# Kulturysta — analiza i porównanie sesji
Notebook używa funkcji projektu; nie duplikuje logiki metryk.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from analysis.comparisons import compare_metric_series
from analysis.metrics import calculate_metrics
from app.models import BoardSample


Wskaż plik CSV sesji (separator `;`, kodowanie UTF-8 z BOM).

In [ ]:
session_file = Path('../data/sessions/UZUPELNIJ/samples.csv')
df = pd.read_csv(session_file, sep=';', encoding='utf-8-sig')
display(df.head())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(df.cop_x, df.cop_y); axes[0].set(title='Stabilogram', xlabel='COP X', ylabel='COP Y'); axes[0].axis('equal')
axes[1].plot(df.timestamp_monotonic, df.cop_x, label='X'); axes[1].plot(df.timestamp_monotonic, df.cop_y, label='Y'); axes[1].legend(); axes[1].set_title('COP w czasie')
axes[2].plot(df.timestamp_monotonic, df.total_weight_kg); axes[2].set_title('Masa w czasie'); plt.show()


In [ ]:
def row_to_sample(row):
    values = row.to_dict()
    values['quality_flags'] = ()
    for key in ('top_left','top_right','bottom_left','bottom_right','total_weight_kg','cop_x','cop_y','filtered_cop_x','filtered_cop_y'):
        if pd.isna(values.get(key)): values[key] = None
    return BoardSample(**{k: values[k] for k in BoardSample.__dataclass_fields__})
samples = [row_to_sample(row) for _, row in df.iterrows()]
calculate_metrics(samples, use_filtered=True)


## Porównanie
Wskaż pliki `metadata.json` dwóch lub większej liczby sesji.

In [ ]:
import json

metadata_files = [
    Path('../data/sessions/SESJA_1/metadata.json'),
    Path('../data/sessions/SESJA_2/metadata.json'),
]
records = [json.loads(path.read_text(encoding='utf-8')) for path in metadata_files]
comparison = compare_metric_series(
    records,
    ['path_length', 'mean_speed', 'rms_cop', 'confidence_ellipse_95_area'],
)
pd.DataFrame(comparison)
